# SCOPE Position LPM Model - Final

This notebook runs the **separate** position-focused M0-M6 analysis. It does not
modify or mix with the governed D0-FE4 content model. The estimand is
`P(cited = 1 | source surfaced in this audit, measurable content)`.

All estimates are observational adjusted associations, not causal effects.

## Numeric-evidence definition

The primary continuous numeric feature is:

`numeric_evidence_total_density = numeric_evidence_total_count / total_main_content_tokens * 1,000`

It is standardized over the row-level analysis sample using the sample standard
deviation (`ddof=1`) to create `z_numeric_evidence_total_density`, which enters
M4 and M5. The position extension is
`numeric_evidence_early_share = numeric_evidence_early_count / numeric_evidence_total_count`.
Early share is `NaN`, not zero, when total numeric evidence is zero, and it does
not enter primary M5.

## Six-class taxonomy controls

Detailed Gemini labels are preserved for QA. M0-M5 use these page-type controls:

- `blog_guide_or_editorial`
- `directory_or_listing`
- `commercial_product_or_service`
- `comparison_or_review`
- `landing_contact_or_support`
- `other_page_function`

The source-type controls are:

- `official_company_or_brand`
- `marketplace_or_directory_platform`
- `blog_or_news_publisher`
- `review_or_community_platform`
- `government_or_public_institution`
- `other_or_unknown`

Source type is made domain-stable using the modal collapsed class across unique
URLs. Exact ties become `other_or_unknown`; low-agreement domains remain flagged
in `position_model_source_type_domain_audit.csv`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
if not (REPO / 'src').exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO = next(path for path in candidates if (path / 'src').exists())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
OUTPUT = REPO / 'outputs/position_model_v1'

from src.econometrics_eda_v2.position_model import run_position_model
manifest = run_position_model(REPO, OUTPUT)
display(pd.DataFrame([manifest]).T.rename(columns={0: 'value'}))

/Volumes/ExtremeSD/Metier/Research/CiteScope-content-audit/.venv/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


/Volumes/ExtremeSD/Metier/Research/CiteScope-content-audit/.venv/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


,value
version,position_model_v3_20260804_taxonomy_6
status,complete
research_question,Among surfaced webpages with measurable conten...
causal_interpretation,False
rows,5264
urls,2600
domains,541
prompts,498
citation_rate,0.324468
M5_rows,3657


## 1. Sample and extraction flow

In [2]:
dataset = pd.read_parquet(OUTPUT / 'position_model_dataset.parquet')
flow = pd.read_csv(OUTPUT / 'position_model_sample_flow.csv')
audit = pd.read_csv(OUTPUT / 'position_model_feature_audit.csv')
display(flow)
display(audit)

,stage_order,stage,n_rows,rows_lost_from_previous,cited_rows,citation_rate,n_urls,n_domains,n_prompts
0,0,all_selected_surfaced_rows,5264,0,1708,0.324468,2600,541,498
1,1,controls_complete,5264,0,1708,0.324468,2600,541,498
2,2,gemini_semantic_measured,3682,1582,1142,0.310158,1911,394,490
3,3,table_placement_measured,4764,-1082,1499,0.314652,2433,496,495
4,4,numeric_total_density_measured,3682,1082,1142,0.310158,1911,394,490
5,5,numeric_early_share_measured,3092,590,944,0.305304,1562,303,488
6,6,M5_joint_complete_case,3657,-565,1138,0.311184,1898,389,490
7,7,M5_high_quality_extraction,3286,371,1028,0.312842,1707,335,488


,feature,n_rows,measured_rows,measured_rate,missing_or_unmeasured_rows,missing_or_unmeasured_rate,status_success_present,status_success_absent,status_scrape_failure,status_main_content_failure,status_parser_failure,status_ineligible,status_ambiguous,manual_validation_precision,manual_validation_recall,manual_validation_status
0,direct_answer_placement,5264,3682,0.699468,1582,0.300532,700,2982,0,1376,4,202,0,NaN,NaN,Gemini semantic detections manually spot-check...
1,table_placement,5264,4764,0.905015,500,0.094985,961,3803,0,202,0,0,298,NaN,NaN,HTML table presence previously audited; placem...
2,question_heading_placement,5264,3682,0.699468,1582,0.300532,1301,2381,0,1376,4,202,0,NaN,NaN,Gemini semantic detections manually spot-check...
3,z_numeric_evidence_total_density,5264,3682,0.699468,1582,0.300532,3092,590,0,1376,4,202,0,NaN,NaN,Gemini semantic detections manually spot-check...
4,numeric_evidence_early_share,5264,3682,0.699468,1582,0.300532,3092,590,0,1376,4,202,0,NaN,NaN,Gemini semantic detections manually spot-check...
5,external_source_placement,5264,5062,0.961626,202,0.038374,1711,3351,0,202,0,0,0,NaN,NaN,Gemini semantic detections manually spot-check...


## 2. Feature coverage and imbalance

In [3]:
coverage = pd.read_csv(OUTPUT / 'position_model_feature_coverage.csv')
categorical = coverage[coverage['n_rows'].notna()].copy()
display(categorical)
px.bar(categorical, x='category', y='n_rows', color='feature', barmode='group',
       title='Position-feature category counts').show()

,feature,feature_label,category,eligible_observations,n_rows,category_share,cited_rows,non_cited_rows,citation_rate,ci_lower,ci_upper,n_domains,n_prompts,sparse_n_lt_20,sparse_cited_lt_5,sparse_non_cited_lt_5,share_lt_5pct,numeric_value
0,direct_answer_placement,Direct-answer placement,direct_answer_early,3682,223.0,0.060565,74.0,149.0,0.331839,0.273344,0.396029,57,152,False,False,False,False,NaN
1,direct_answer_placement,Direct-answer placement,direct_answer_late,3682,477.0,0.129549,191.0,286.0,0.400419,0.357412,0.445017,66,253,False,False,False,False,NaN
2,direct_answer_placement,Direct-answer placement,no_direct_answer,3682,2982.0,0.809886,877.0,2105.0,0.294098,0.278018,0.310708,348,488,False,False,False,False,NaN
3,table_placement,Table placement,no_table,4764,3803.0,0.798279,1171.0,2632.0,0.307915,0.293443,0.322774,435,495,False,False,False,False,NaN
4,table_placement,Table placement,table_early,4764,820.0,0.172124,269.0,551.0,0.328049,0.296780,0.360921,77,330,False,False,False,False,NaN
5,table_placement,Table placement,table_late,4764,141.0,0.029597,59.0,82.0,0.418440,0.340237,0.500969,32,116,False,False,False,True,NaN
6,question_heading_placement,Question-heading placement,no_question_heading,3682,2381.0,0.646659,644.0,1737.0,0.270475,0.253013,0.288676,297,464,False,False,False,False,NaN
7,question_heading_placement,Question-heading placement,question_heading_early,3682,736.0,0.199891,294.0,442.0,0.399457,0.364682,0.435275,113,312,False,False,False,False,NaN
8,question_heading_placement,Question-heading placement,question_heading_late,3682,565.0,0.153449,204.0,361.0,0.361062,0.322519,0.401482,79,280,False,False,False,False,NaN
9,external_source_placement,External-source placement,external_source_early,5062,923.0,0.182339,291.0,632.0,0.315276,0.286120,0.345964,124,377,False,False,False,False,NaN


In [4]:
fig = go.Figure()
for feature, group in categorical.groupby('feature'):
    fig.add_trace(go.Scatter(
        x=group['category'], y=group['citation_rate'], mode='markers+lines', name=feature,
        error_y=dict(type='data', symmetric=False,
                     array=group['ci_upper'] - group['citation_rate'],
                     arrayminus=group['citation_rate'] - group['ci_lower']),
        text=group['n_rows'].map(lambda n: f'n={n:,}'),
    ))
fig.update_layout(title='Citation rate by placement with Wilson 95% CI', yaxis_tickformat='.0%')
fig.show()

## 3. Domain, prompt, and within-domain support

In [5]:
domain = pd.read_csv(OUTPUT / 'position_model_domain_concentration.csv')
prompt = pd.read_csv(OUTPUT / 'position_model_prompt_concentration.csv')
within = pd.read_csv(OUTPUT / 'position_model_within_domain_variation.csv')
clusters = pd.read_csv(OUTPUT / 'position_model_cluster_support.csv')
display(domain)
display(prompt)
display(within)
display(clusters)

,feature,category,dimension,n_rows,groups_represented,top_group_share,top_five_group_share,hhi,effective_groups,median_observations_per_group,singleton_observation_share,concentration_flag,top_contributors
0,direct_answer_placement,direct_answer_early,domain,223,57,0.152466,0.426009,0.056486,17.703453,1.0,0.130045,not_flagged,condoreviewsthailand.com:34; thailandcondoshop...
1,direct_answer_placement,direct_answer_late,domain,477,66,0.171908,0.467505,0.065077,15.366313,2.0,0.050314,not_flagged,hipflat.co.th:82; kant.co.th:49; tvc.co.th:42;...
2,direct_answer_placement,no_direct_answer,domain,2982,348,0.108317,0.303488,0.029380,34.036822,2.0,0.051308,low_effective_group_count,ddproperty.com:323; propertyhub.in.th:245; bkk...
3,table_placement,no_table,domain,3803,435,0.108861,0.272417,0.025516,39.190887,2.0,0.047068,low_effective_group_count,superagent.co:414; ddproperty.com:259; bkkcond...
4,table_placement,table_early,domain,820,77,0.181707,0.557317,0.083305,12.004142,2.0,0.040244,low_effective_group_count,condoreviewsthailand.com:149; propertyhub.in.t...
5,table_placement,table_late,domain,141,32,0.219858,0.595745,0.098637,10.138195,1.5,0.113475,not_flagged,o-waw.com:31; condoreviewsthailand.com:20; pro...
6,question_heading_placement,no_question_heading,domain,2381,297,0.133977,0.367493,0.040082,24.948669,2.0,0.053759,low_effective_group_count,ddproperty.com:319; propertyhub.in.th:244; bkk...
7,question_heading_placement,question_heading_early,domain,736,113,0.066576,0.254076,0.026210,38.152979,2.0,0.052989,not_flagged,origin.co.th:49; thailandcondoshop.com:44; kan...
8,question_heading_placement,question_heading_late,domain,565,79,0.201770,0.437168,0.069478,14.393120,3.0,0.051327,low_effective_group_count,condoreviewsthailand.com:114; connex.in.th:70;...
9,external_source_placement,external_source_early,domain,923,124,0.089924,0.352113,0.037852,26.418861,2.0,0.056338,not_flagged,hipflat.co.th:83; 9asset.com:69; livinginsider...


,feature,category,dimension,n_rows,groups_represented,top_group_share,top_five_group_share,hhi,effective_groups,median_observations_per_group,singleton_observation_share,concentration_flag,top_contributors
0,direct_answer_placement,direct_answer_early,prompt,223,152,0.017937,0.076233,0.008064,124.012469,1.0,0.434978,not_flagged,AREA_CONDO_NB_500_238:4; AREA_CONDO_NB_500_444...
1,direct_answer_placement,direct_answer_late,prompt,477,253,0.016771,0.071279,0.005990,166.932502,1.0,0.306080,not_flagged,AREA_CONDO_NB_500_128:8; AREA_CONDO_NB_500_110...
2,direct_answer_placement,no_direct_answer,prompt,2982,488,0.006372,0.023139,0.002487,402.112870,6.0,0.008048,not_flagged,AREA_CONDO_NB_500_408:19; AREA_CONDO_NB_500_40...
3,table_placement,no_table,prompt,3803,495,0.004470,0.018932,0.002280,438.519420,8.0,0.002104,not_flagged,AREA_CONDO_NB_500_408:17; AREA_CONDO_NB_500_20...
4,table_placement,table_early,prompt,820,330,0.009756,0.045122,0.004075,245.401460,2.0,0.123171,not_flagged,AREA_CONDO_NB_500_390:8; AREA_CONDO_NB_500_248...
5,table_placement,table_late,prompt,141,116,0.021277,0.092199,0.009909,100.918782,1.0,0.666667,not_flagged,AREA_CONDO_NB_500_223:3; AREA_CONDO_NB_500_280...
6,question_heading_placement,no_question_heading,prompt,2381,464,0.007560,0.028559,0.002866,348.978824,5.0,0.018060,not_flagged,AREA_CONDO_NB_500_408:18; AREA_CONDO_NB_500_40...
7,question_heading_placement,question_heading_early,prompt,736,312,0.010870,0.050272,0.004567,218.955538,2.0,0.165761,not_flagged,AREA_CONDO_NB_500_040:8; AREA_CONDO_NB_500_083...
8,question_heading_placement,question_heading_late,prompt,565,280,0.019469,0.074336,0.005329,187.669018,2.0,0.237168,not_flagged,AREA_CONDO_NB_500_481:11; AREA_CONDO_NB_500_48...
9,external_source_placement,external_source_early,prompt,923,377,0.010834,0.045504,0.003741,267.313775,2.0,0.137595,not_flagged,AREA_CONDO_NB_500_361:10; AREA_CONDO_NB_500_12...


,feature,eligible_rows,domains_total,domains_with_at_least_two_pages,domains_with_presence_variation,domains_with_early_late_variation,domains_with_outcome_variation,domains_with_placement_and_outcome_variation,observations_in_informative_domains,within_domain_standard_deviation,between_domain_standard_deviation,fixed_effect_readiness,readiness_rule
0,direct_answer_placement,3682,394,230,54,54,142,48,1981,0.236655,0.345004,Ready,Ready >=30 varying and >=20 informative; cauti...
1,table_placement,4764,496,289,41,41,176,39,1954,0.226428,0.345665,Ready,Ready >=30 varying and >=20 informative; cauti...
2,question_heading_placement,3682,394,230,62,62,142,60,2480,0.265671,0.434572,Ready,Ready >=30 varying and >=20 informative; cauti...
3,external_source_placement,5062,528,306,76,76,192,66,2749,0.267669,0.432499,Ready,Ready >=30 varying and >=20 informative; cauti...


,dimension,n_clusters,median_observations,minimum_observations,maximum_observations,singleton_cluster_share,singleton_observation_share
0,domain,389,2.0,1,326,0.416452,0.044299
1,prompt,490,8.0,1,19,0.028571,0.003828


## 4. M0-M6 model results

In [6]:
results = pd.read_csv(OUTPUT / 'position_model_results_long.csv')
primary = results[results['is_primary_inference'].astype(bool)].copy()
focal = primary[primary['term'].str.contains('placement|numeric_evidence', case=False, regex=True)]
display(focal[['model_id','term','estimate_pp','ci_lower_pp','ci_upper_pp','p_value','bh_q_value',
               'n_obs','n_cited','n_domains','n_prompts','se_method']])

plot = focal[focal['model_id'].eq('M5')].copy()
fig = go.Figure(go.Scatter(
    x=plot['estimate_pp'], y=plot['term'], mode='markers',
    error_x=dict(type='data', symmetric=False,
                 array=plot['ci_upper_pp'] - plot['estimate_pp'],
                 arrayminus=plot['estimate_pp'] - plot['ci_lower_pp']),
))
fig.add_vline(x=0, line_dash='dash')
fig.update_layout(title='M5 adjusted associations', xaxis_title='Percentage points')
fig.show()

,model_id,term,estimate_pp,ci_lower_pp,ci_upper_pp,p_value,bh_q_value,n_obs,n_cited,n_domains,n_prompts,se_method
3546,M1,"C(direct_answer_placement, Treatment(reference...",-5.026188,-14.663051,4.610675,0.306669,0.306669,3682,1142,394,490,two_way_cluster_domain_prompt
3547,M1,"C(direct_answer_placement, Treatment(reference...",7.018181,-2.732301,16.768663,0.158322,0.277063,3682,1142,394,490,two_way_cluster_domain_prompt
5573,M2,"C(table_placement, Treatment(reference='no_tab...",4.213036,-2.875707,11.301779,0.244075,0.306669,4764,1499,496,495,two_way_cluster_domain_prompt
5574,M2,"C(table_placement, Treatment(reference='no_tab...",9.564018,-2.220366,21.348401,0.111683,0.260593,4764,1499,496,495,two_way_cluster_domain_prompt
7590,M3,"C(question_heading_placement, Treatment(refere...",7.587104,-0.552019,15.726228,0.067695,0.251923,3682,1142,394,490,two_way_cluster_domain_prompt
7591,M3,"C(question_heading_placement, Treatment(refere...",6.441597,-0.575353,13.458546,0.071978,0.251923,3682,1142,394,490,two_way_cluster_domain_prompt
10098,M4,z_numeric_evidence_total_density,1.164776,-0.967562,3.297113,0.284341,0.306669,3682,1142,394,490,two_way_cluster_domain_prompt
11625,M5,"C(direct_answer_placement, Treatment(reference...",-8.265202,-18.844564,2.314159,0.125711,NaN,3657,1138,389,490,two_way_cluster_domain_prompt
11626,M5,"C(direct_answer_placement, Treatment(reference...",5.131522,-4.649309,14.912353,0.303810,NaN,3657,1138,389,490,two_way_cluster_domain_prompt
11627,M5,"C(table_placement, Treatment(reference='no_tab...",1.945692,-4.380215,8.271598,0.546618,NaN,3657,1138,389,490,two_way_cluster_domain_prompt


## 5. Multicollinearity and confidence-interval diagnostics

In [7]:
multi = pd.read_csv(OUTPUT / 'position_model_multicollinearity.csv')
ci = pd.read_csv(OUTPUT / 'position_model_ci_diagnostics.csv')
display(multi[multi['row_type'].eq('vif')].sort_values('vif', ascending=False))
display(ci[['model_id','term','ci_width_pp','category_sample_size','category_cited_count',
            'maximum_domain_share','maximum_prompt_share','vif',
            'leave_one_domain_out_min_pp','leave_one_domain_out_max_pp',
            'grounded_ci_explanation']])

,row_type,variable,related_variable,association,vif,condition_number,warning
8,vif,"C(page_type_model_6, Treatment(reference='blog...",NaN,NaN,2.281133,3.315814,not_flagged
13,vif,"C(source_type_model_6, Treatment(reference='of...",NaN,NaN,2.226538,3.315814,not_flagged
6,vif,"C(page_type_model_6, Treatment(reference='blog...",NaN,NaN,1.861940,3.315814,not_flagged
7,vif,"C(page_type_model_6, Treatment(reference='blog...",NaN,NaN,1.583371,3.315814,not_flagged
15,vif,"C(source_type_model_6, Treatment(reference='of...",NaN,NaN,1.551770,3.315814,not_flagged
4,vif,"C(question_heading_placement, Treatment(refere...",NaN,NaN,1.545509,3.315814,not_flagged
11,vif,"C(source_type_model_6, Treatment(reference='of...",NaN,NaN,1.402109,3.315814,not_flagged
17,vif,log_word_count,NaN,NaN,1.289444,3.315814,not_flagged
5,vif,"C(question_heading_placement, Treatment(refere...",NaN,NaN,1.277132,3.315814,not_flagged
0,vif,"C(direct_answer_placement, Treatment(reference...",NaN,NaN,1.268844,3.315814,not_flagged


,model_id,term,ci_width_pp,category_sample_size,category_cited_count,maximum_domain_share,maximum_prompt_share,vif,leave_one_domain_out_min_pp,leave_one_domain_out_max_pp,grounded_ci_explanation
0,M2,"C(table_placement, Treatment(reference='no_tab...",23.568767,141.0,59.0,0.219858,0.021277,1.104639,2.547265,7.800960,This interval is wide because the category rep...
1,M5,"C(table_placement, Treatment(reference='no_tab...",22.376811,141.0,59.0,0.219858,0.021277,1.104639,2.547265,7.800960,This interval is wide because the category rep...
2,M5,"C(direct_answer_placement, Treatment(reference...",21.158723,223.0,74.0,0.152466,0.017937,1.268844,-11.473914,-5.975711,"This interval is wide because cell support, cl..."
3,M5,"C(direct_answer_placement, Treatment(reference...",19.561662,477.0,191.0,0.171908,0.016771,1.171077,1.642648,8.058732,This interval is comparatively narrow because ...
4,M1,"C(direct_answer_placement, Treatment(reference...",19.500964,477.0,191.0,0.171908,0.016771,1.171077,1.642648,8.058732,This interval is comparatively narrow because ...
5,M1,"C(direct_answer_placement, Treatment(reference...",19.273726,223.0,74.0,0.152466,0.017937,1.268844,-11.473914,-5.975711,This interval is comparatively narrow because ...
6,M5,"C(question_heading_placement, Treatment(refere...",16.780869,736.0,294.0,0.066576,0.010870,1.545509,5.317185,8.800573,This interval is comparatively narrow because ...
7,M5,"C(question_heading_placement, Treatment(refere...",16.327372,565.0,204.0,0.201770,0.019469,1.277132,2.312665,5.523320,This interval is comparatively narrow because ...
8,M3,"C(question_heading_placement, Treatment(refere...",16.278247,736.0,294.0,0.066576,0.010870,1.545509,5.317185,8.800573,This interval is comparatively narrow because ...
9,M2,"C(table_placement, Treatment(reference='no_tab...",14.177486,820.0,269.0,0.181707,0.009756,1.216655,-0.080292,3.764165,This interval is comparatively narrow because ...


## 6. Stability and robustness

In [8]:
influence = pd.read_csv(OUTPUT / 'position_model_influence_diagnostics.csv')
robustness = pd.read_csv(OUTPUT / 'position_model_robustness_results.csv')
predicted = pd.read_csv(OUTPUT / 'position_model_predicted_probability_diagnostics.csv')
display(robustness)
display(predicted)

m5_term = focal.loc[focal['model_id'].eq('M5'), 'term'].iloc[0]
stability = influence[(influence['term'].eq(m5_term)) &
                      (influence['influence_dimension'].eq('source_root_domain'))]
px.scatter(stability, x='removed_group', y='estimate_pp',
           hover_data=['removed_rows','change_pp','sign_changed'],
           title=f'Leave-one-domain-out: {m5_term}').show()

,model_id,term,estimate,robustness_type,n_obs,notes,formula,estimate_pp,standard_error,ci_lower,...,n_cited,n_domains,n_prompts,domain_clusters,prompt_clusters,adjusted_r_squared,fixed_effects,model_matrix_rank,model_matrix_columns,z
0,R1_direct_answer_continuous_position,model_status,NaN,continuous_position_conditional_on_feature_pre...,700,not estimable: ValueError: R1_direct_answer_co...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,R2_direct_answer_quartile_position,Intercept,1.093079,Q1_Q4_placement,3682,NaN,cited ~ C(direct_answer_position_quartile_mode...,109.307940,0.207924,0.685556,...,1142.0,394.0,490.0,394.0,490.0,0.080875,prompt_id,505.0,505.0,NaN
2,R2_direct_answer_quartile_position,"C(direct_answer_position_quartile_model, Treat...",-0.091831,Q1_Q4_placement,3682,NaN,cited ~ C(direct_answer_position_quartile_mode...,-9.183101,0.047378,-0.184691,...,1142.0,394.0,490.0,394.0,490.0,0.080875,prompt_id,505.0,505.0,NaN
3,R2_direct_answer_quartile_position,"C(direct_answer_position_quartile_model, Treat...",0.020083,Q1_Q4_placement,3682,NaN,cited ~ C(direct_answer_position_quartile_mode...,2.008321,0.064511,-0.106355,...,1142.0,394.0,490.0,394.0,490.0,0.080875,prompt_id,505.0,505.0,NaN
4,R2_direct_answer_quartile_position,"C(direct_answer_position_quartile_model, Treat...",0.023054,Q1_Q4_placement,3682,NaN,cited ~ C(direct_answer_position_quartile_mode...,2.305362,0.055564,-0.085850,...,1142.0,394.0,490.0,394.0,490.0,0.080875,prompt_id,505.0,505.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12180,R4_logit_average_marginal_effects,C(prompt_id)[T.AREA_CONDO_NB_500_498],0.076936,logit_average_marginal_effects,3657,NaN,NaN,7.693562,0.309130,-0.528948,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.248878
12181,R4_logit_average_marginal_effects,C(prompt_id)[T.AREA_CONDO_NB_500_499],0.148909,logit_average_marginal_effects,3657,NaN,NaN,14.890885,0.301518,-0.442055,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.493864
12182,R4_logit_average_marginal_effects,C(prompt_id)[T.AREA_CONDO_NB_500_500],0.120242,logit_average_marginal_effects,3657,NaN,NaN,12.024169,0.380474,-0.625474,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.316031
12183,R4_logit_average_marginal_effects,z_numeric_evidence_total_density,0.015880,logit_average_marginal_effects,3657,NaN,NaN,1.587961,0.008341,-0.000469,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.903779


,model_id,n_obs,minimum_predicted_probability,maximum_predicted_probability,n_below_zero,share_below_zero,n_above_one,share_above_one
0,M0,5264,-0.164319,1.336249,67,0.012728,47,0.008929
1,M1,3682,-0.185760,1.242271,177,0.048072,37,0.010049
2,M2,4764,-0.181668,1.293953,86,0.018052,45,0.009446
3,M3,3682,-0.179267,1.217137,178,0.048343,36,0.009777
4,M4,3682,-0.189534,1.247750,169,0.045899,35,0.009506
5,M5,3657,-0.197102,1.267933,189,0.051682,34,0.009297


## 7. Interpretation boundary

- Coefficients are percentage-point adjusted associations among surfaced sources.
- A coefficient whose interval includes zero is retained as an inconclusive/null result.
- Placement categories can reflect domain templates, page function, prompt mix, or extraction behavior.
- Stability across M5, alternative SEs, logit AMEs, and leave-one-domain-out checks improves reportability but does not establish causality.
- Moving a feature earlier on a webpage is **not** proven to change citation probability.
- External-source placement remains descriptive because formal placement-specific manual validation is unavailable.

In [9]:
display(Markdown('```text\n' + (OUTPUT / 'POSITION_MODEL_FINDINGS.txt').read_text() + '\n```'))

```text
POSITION MODEL FINDINGS
=======================

Research estimand: P(cited = 1 | source surfaced in this audit, measurable content).
All estimates are adjusted observational associations, not causal effects.

Rows: 5,264; URLs: 2,600; domains: 541; prompts: 498.
Primary inference: two_way_cluster_domain_prompt.
External-source M6 gate: all_cells_n_ge_20=pass; all_cells_share_ge_5pct=pass; no_top_domain_gt_25pct=pass; formal_manual_validation_available=fail.
Page-type control: page_type_model_6 (6 classes); detailed Gemini labels preserved.
Source-type control: source_type_model_6 (6 classes); detailed Gemini labels preserved.
Source domain-consensus audit: 18 low-confidence domains; 14 tied domains assigned other_or_unknown.

1. Adequate category support: direct_answer_placement, table_placement, question_heading_placement, external_source_placement.
2. Imbalanced cells: [{'feature': 'table_placement', 'category': 'table_late'}]
3. Domain concentration flags: [{'feature': 'direct_answer_placement', 'category': 'no_direct_answer', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'table_placement', 'category': 'no_table', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'table_placement', 'category': 'table_early', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'question_heading_placement', 'category': 'no_question_heading', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'question_heading_placement', 'category': 'question_heading_late', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'external_source_placement', 'category': 'external_source_late', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'external_source_placement', 'category': 'no_external_source', 'concentration_flag': 'low_effective_group_count'}, {'feature': 'numeric_evidence_early_share', 'category': 'top_10_percent', 'concentration_flag': 'low_effective_group_count'}]
4. Prompt concentration flags: none
5. Adequate within-domain variation: {'direct_answer_placement': 'Ready', 'table_placement': 'Ready', 'question_heading_placement': 'Ready', 'external_source_placement': 'Ready'}

Primary M5 estimates (percentage points):
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: -8.27 pp (95% CI -18.84, 2.31; p=0.1257).
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: 5.13 pp (95% CI -4.65, 14.91; p=0.3038).
- C(table_placement, Treatment(reference='no_table'))[T.table_early]: 1.95 pp (95% CI -4.38, 8.27; p=0.5466).
- C(table_placement, Treatment(reference='no_table'))[T.table_late]: 5.97 pp (95% CI -5.22, 17.15; p=0.296).
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: 7.89 pp (95% CI -0.50, 16.28; p=0.0653).
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: 4.28 pp (95% CI -3.89, 12.44; p=0.3044).
- z_numeric_evidence_total_density: 1.45 pp (95% CI -0.69, 3.58; p=0.1833).

6. Coefficients with the widest confidence intervals and 7. evidence-based reasons:
- M2 C(table_placement, Treatment(reference='no_table'))[T.table_late]: This interval is wide because the category represents only 3.0% of eligible rows.
- M5 C(table_placement, Treatment(reference='no_table'))[T.table_late]: This interval is wide because the category represents only 3.0% of eligible rows.
- M5 C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: This interval is wide because cell support, cluster support, concentration, and VIF are not individually severe.
- M5 C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: This interval is comparatively narrow because two-way clustered uncertainty is much larger than HC3.
- M1 C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: This interval is comparatively narrow because two-way clustered uncertainty is much larger than HC3.
- M1 C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: This interval is comparatively narrow because cell support, cluster support, concentration, and VIF are not individually severe.
- M5 C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: This interval is comparatively narrow because two-way clustered uncertainty is much larger than HC3.
- M5 C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: This interval is comparatively narrow because cell support, cluster support, concentration, and VIF are not individually severe.
- M3 C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: This interval is comparatively narrow because two-way clustered uncertainty is much larger than HC3.
- M2 C(table_placement, Treatment(reference='no_table'))[T.table_early]: This interval is comparatively narrow because two-way clustered uncertainty is much larger than HC3; the sign changes in leave-one-domain-out analysis.

8. Influential-domain sensitivity:
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: 0 LODO sign changes; maximum absolute change 3.21 pp
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: 0 LODO sign changes; maximum absolute change 3.49 pp
- C(table_placement, Treatment(reference='no_table'))[T.table_early]: 1 LODO sign changes; maximum absolute change 2.03 pp
- C(table_placement, Treatment(reference='no_table'))[T.table_late]: 0 LODO sign changes; maximum absolute change 3.42 pp
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: 0 LODO sign changes; maximum absolute change 2.57 pp
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: 0 LODO sign changes; maximum absolute change 1.96 pp
- z_numeric_evidence_total_density: 0 LODO sign changes; maximum absolute change 0.63 pp

9. Separate-model to M5 stability:
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: same sign from separate model to M5
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: same sign from separate model to M5
- C(table_placement, Treatment(reference='no_table'))[T.table_early]: same sign from separate model to M5
- C(table_placement, Treatment(reference='no_table'))[T.table_late]: same sign from separate model to M5
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: same sign from separate model to M5
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: same sign from separate model to M5
- z_numeric_evidence_total_density: same sign from separate model to M5

10. Alternative-standard-error stability:
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: CI excludes zero under 2/4 SE methods
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: CI excludes zero under 0/4 SE methods
- C(table_placement, Treatment(reference='no_table'))[T.table_early]: CI excludes zero under 0/4 SE methods
- C(table_placement, Treatment(reference='no_table'))[T.table_late]: CI excludes zero under 0/4 SE methods
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: CI excludes zero under 2/4 SE methods
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: CI excludes zero under 0/4 SE methods
- z_numeric_evidence_total_density: CI excludes zero under 0/4 SE methods

11. Logistic-regression AME cross-check:
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: AME -8.01 pp
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: AME 4.76 pp
- C(table_placement, Treatment(reference='no_table'))[T.table_early]: AME 2.34 pp
- C(table_placement, Treatment(reference='no_table'))[T.table_late]: AME 4.56 pp
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: AME 8.37 pp
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: AME 4.65 pp
- z_numeric_evidence_total_density: AME 1.59 pp

12. Domain fixed effects feasibility:
- Feasible after absorbing prompt and domain effects on supported multi-URL domains; page/source taxonomy controls are omitted because they are absorbed or nearly absorbed.
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]: -11.98 pp (95% CI -21.64, -2.33)
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]: 5.89 pp (95% CI -5.14, 16.92)
- C(table_placement, Treatment(reference='no_table'))[T.table_early]: 2.30 pp (95% CI -4.96, 9.55)
- C(table_placement, Treatment(reference='no_table'))[T.table_late]: 14.42 pp (95% CI 3.23, 25.61)
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]: 4.06 pp (95% CI -5.61, 13.73)
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]: 2.69 pp (95% CI -7.72, 13.10)
- z_numeric_evidence_total_density: 2.00 pp (95% CI 0.05, 3.96)

13. Findings sufficiently stable to report as adjusted associations (still non-causal):
- None meet every primary stability screen.

14. Inconclusive findings:
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_early]
- C(direct_answer_placement, Treatment(reference='no_direct_answer'))[T.direct_answer_late]
- C(table_placement, Treatment(reference='no_table'))[T.table_early]
- C(table_placement, Treatment(reference='no_table'))[T.table_late]
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_early]
- C(question_heading_placement, Treatment(reference='no_question_heading'))[T.question_heading_late]
- z_numeric_evidence_total_density

Interpretation boundary:
A null or statistically insignificant result is retained. Moving a feature on a page is not proven to change citation probability.
Domain/template, page-function, prompt, extraction, and selection confounding remain possible.
The external-source detector is descriptive only because formal placement-specific manual validation is unavailable.
```